# US Accidents — Feature Engineering Pipeline

**Pipeline stages:**
1. Session setup & config
2. Load pre-processed data
3. Temporal feature extraction
4. Geospatial bucketing
5. Domain features (road risk, weather flag)
6. Boolean casting
7. Log transform for skewed numerics
8. Categorical encoding (StringIndexer)
9. Class-weight computation (native Spark, no UDF)
10. Train / test split  ← must happen BEFORE fitting any scaler
11. Feature assembly + StandardScaler
12. Save feature-engineered splits

In [1]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, count, date_format
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler,
)

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark

In [4]:
Data_Path = "Data/Full_Data"

In [5]:
df = spark.read \
    .option("mergeSchema", "false") \
    .parquet(Data_Path)

In [6]:
df = df.repartition(16)

In [7]:
df.printSchema()
df.show(5, truncate=False)

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true

In [8]:
df = (
    df
    .withColumn("Hour",      F.hour("Start_Time"))
    .withColumn("DayOfWeek", F.dayofweek("Start_Time"))   # 1=Sun … 7=Sat
    .withColumn("Month",     F.month("Start_Time"))
    # Derived convenience flag — useful for tree-based models.
    .withColumn("IsWeekend",
        F.when(F.dayofweek("Start_Time").isin(1, 7), 1).otherwise(0)
    )
)

df = df.withColumn(
    "Season",
    F.when(F.col("Month").isin(12, 1, 2), "Winter")
     .when(F.col("Month").isin(3,  4, 5), "Spring")
     .when(F.col("Month").isin(6,  7, 8), "Summer")
     .otherwise("Autumn")
)

In [9]:
BIN_SIZE = 0.5

In [10]:
df = (
    df
    .withColumn("Lat_Bucket", (F.col("Start_Lat") / BIN_SIZE).cast("int"))
    .withColumn("Lng_Bucket", (F.col("Start_Lng") / BIN_SIZE).cast("int"))
)

In [11]:
bool_cols = ["Traffic_Signal", "Junction", "Crossing", "Stop"]
for c in bool_cols:
    df = df.withColumn(c, F.col(c).cast("int"))

In [12]:
df = df.withColumn(
    "Road_Risk_Score",
    # coalesce each column to 0 so a single NULL doesn't nullify the whole sum.
    F.coalesce(F.col("Traffic_Signal"), F.lit(0))
    + F.coalesce(F.col("Junction"),       F.lit(0))
    + F.coalesce(F.col("Crossing"),       F.lit(0))
    + F.coalesce(F.col("Stop"),           F.lit(0))
)

df = df.withColumn(
    "Bad_Weather_Flag",
    F.when(
        (F.col("Visibility(mi)")     <  2  ) |
        (F.col("Precipitation(in)")  >  0.1) |
        (F.col("Wind_Speed(mph)")    > 20  ),
        1
    ).otherwise(0)
)

# Log-transform skewed continuous feature.
df = df.withColumn("Log_Distance", F.log1p("Distance(mi)"))

In [13]:
cat_cols = [
    "Weather_Condition",
    "Wind_Direction",
    "Season",
    "Sunrise_Sunset",
    "Civil_Twilight",
    "Nautical_Twilight",
    "Astronomical_Twilight",
    # High-cardinality — keep for tree-based models, remove for linear ones:
    "Street",
    "City",
    "County",
    "State",
]

In [14]:
cat_cols = [c for c in cat_cols if c in df.columns]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

pipeline = Pipeline(stages=indexers)

df = pipeline.fit(df).transform(df)

In [15]:
severity_counts = df.groupBy("Severity").agg(count("*").alias("class_count"))

total      = df.count()
n_classes  = severity_counts.count()

severity_weights = severity_counts.withColumn(
    "weight",
    F.lit(total) / (F.lit(n_classes) * F.col("class_count"))
)

# Join back so every row gets its class weight — no UDF, fully distributed.
df = df.join(severity_weights.select("Severity", "weight"), on="Severity", how="left")

df.groupBy("Severity").agg(
    count("*").alias("count"),
    F.first("weight").alias("weight")
).orderBy("Severity").show()

+--------+-------+-------------------+
|Severity|  count|             weight|
+--------+-------+-------------------+
|       1|  93419| 28.296534966120383|
|       2|8689972|0.30419361535342115|
|       3|1454442|  1.817490143986491|
|       4| 335903|  7.869634983909045|
+--------+-------+-------------------+



In [16]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train_df.count():,}  |  Test rows: {test_df.count():,}")

Train rows: 8,459,336  |  Test rows: 2,115,303


In [17]:
df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- ID: string (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true

In [18]:
feature_cols = [

    # =========================
    # 📍 GEO FEATURES
    # =========================
    "Start_Lat", "Start_Lng",
    "Lat_Bucket", "Lng_Bucket",

    # =========================
    # 📊 NUMERIC FEATURES
    # =========================
    "Distance(mi)", "Log_Distance",
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",

    # =========================
    # 🚦 ROAD / BINARY FEATURES
    # =========================
    "Traffic_Signal", "Junction", "Crossing", "Stop",
    "Amenity", "Bump", "Give_Way", "No_Exit",
    "Railway", "Roundabout", "Station", "Traffic_Calming", "Turning_Loop",

    # =========================
    # ⏱ TIME FEATURES
    # =========================
    "Hour", "DayOfWeek", "Month", "IsWeekend",

    # =========================
    # 🧠 ENGINEERED FEATURES
    # =========================
    "Road_Risk_Score",
    "Bad_Weather_Flag",

    # =========================
    # 🧾 ENCODED CATEGORICALS
    # =========================
    "Weather_Condition_idx",
    "Wind_Direction_idx",
    "Season_idx",
    "Sunrise_Sunset_idx",
    "Civil_Twilight_idx",
    "Nautical_Twilight_idx",
    "Astronomical_Twilight_idx",

    # =========================
    # 🌍 OPTIONAL (USE CAREFULLY)
    # =========================
    # "City_idx",
    # "State_idx",
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)

train_assembled = assembler.transform(train_df)
test_assembled  = assembler.transform(test_df)

In [19]:
SCALE_FEATURES = False   # set True only when needed

if SCALE_FEATURES:
    scaler = StandardScaler(
        inputCol="features",
        outputCol="scaled_features",
        withMean=True,   # densifies sparse vectors — OK here (all features are numeric)
        withStd=True,
    )
    scaler_model    = scaler.fit(train_assembled)      # fit on train only!
    train_assembled = scaler_model.transform(train_assembled)
    test_assembled  = scaler_model.transform(test_assembled)
    final_feature_col = "scaled_features"
else:
    final_feature_col = "features"

print(f"Using feature column: '{final_feature_col}'")

Using feature column: 'features'


In [20]:
OUTPUT_BASE = "Data/Feature_Engineered_Data"

In [21]:
def save_split(df_split, path):
    """Format Start_Time and write as Parquet."""
    (
        df_split
        .withColumn("Start_Time", date_format(col("Start_Time"), "yyyy-MM-dd HH:mm:ss"))
        .write
        .mode("overwrite")
        .parquet(path)
    )
    print(f"Saved → {path}")

save_split(train_assembled, f"{OUTPUT_BASE}/train")
save_split(test_assembled,  f"{OUTPUT_BASE}/test")

Saved → Data/Feature_Engineered_Data/train
Saved → Data/Feature_Engineered_Data/test


In [22]:
train_reloaded = spark.read.parquet(f"{OUTPUT_BASE}/train")
print("Train schema after reload:")
train_reloaded.printSchema()
print()
train_reloaded.select("Severity", "weight", "features").show(5, truncate=True)

Train schema after reload:
root
 |-- Severity: integer (nullable = true)
 |-- ID: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi):